In [91]:
import pandas as pd
import seaborn as sns

df = sns.load_dataset("titanic")

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [93]:
# The Titanic dataset already has:

# sibsp (Number of siblings/spouses)

# and

# parch (Number of parents/children)

# Instead of keeping them separate:

df["family_size"] = df["sibsp"] + df["parch"] + 1

# Why +1?

# Because the passenger counts too.

In [95]:
df[["sibsp","parch","family_size"]].head()

,sibsp,parch,family_size
0,1,0,2
1,1,0,2
2,0,0,1
3,1,0,2
4,0,0,1


In [97]:
# Does being alone matter?

# Instead of making the model figure it out...

# we tell it directly.

# Family Size = 1

# ↓

# is_alone = 1

# Otherwise:

# Family Size > 1

# ↓

# is_alone = 0

df["is_alone"] = (
    df["family_size"] == 1
).astype(int)

In [99]:
df[["family_size","is_alone"]].head()

,family_size,is_alone
0,2,0
1,2,0
2,1,1
3,2,0
4,1,1


In [101]:
df.groupby("is_alone")["survived"].mean()

is_alone
0    0.505650
1    0.303538
Name: survived, dtype: float64

In [103]:
# Look at this column:

# df["fare"]

# Imagine:

# Family of 5

# Fare = £100

# Did each person pay:

# £100

# ?

# Probably not.

# Each effectively paid:

# £20

df["fare_per_person"] = (
    df["fare"] /
    df["family_size"]
)

In [105]:
# Look at Age.

# Which is more informative?

# Age = 29

# or

# Adult

# Sometimes categories are easier for models to learn.

# We can create:

df["age_group"] = pd.cut(
    df["age"],
    bins=[0,12,18,35,60,100],
    labels=[
        "Child",
        "Teen",
        "Young Adult",
        "Adult",
        "Senior"
    ]
)

In [107]:
df = pd.get_dummies(
    df,
    columns=["age_group"],
    drop_first=True
)

In [109]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,...,embark_town,alive,alone,family_size,is_alone,fare_per_person,age_group_Teen,age_group_Young Adult,age_group_Adult,age_group_Senior
0,0,3,male,22.0,1,0,7.2500,S,Third,man,...,Southampton,no,False,2,0,3.62500,False,True,False,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,...,Cherbourg,yes,False,2,0,35.64165,False,False,True,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,...,Southampton,yes,True,1,1,7.92500,False,True,False,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,...,Southampton,yes,False,2,0,26.55000,False,True,False,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,...,Southampton,no,True,1,1,8.05000,False,True,False,False


In [111]:
# TARGET
y = df["survived"]

In [113]:
df.drop(columns=["class"], inplace=True)

In [115]:
df["who"].value_counts()

who
man      537
woman    271
child     83
Name: count, dtype: int64

In [117]:
df.drop(columns=["who"], inplace=True)

In [119]:
df.drop(columns=["adult_male"], inplace=True)

In [121]:
df.drop(columns=["deck"], inplace=True)

In [123]:
df[["embarked","embark_town"]].head()

,embarked,embark_town
0,S,Southampton
1,C,Cherbourg
2,S,Southampton
3,S,Southampton
4,S,Southampton


In [125]:
df.drop(columns=["embark_town"], inplace=True)

In [127]:
df.drop(columns=["alive"], inplace=True)

In [129]:
df.drop(columns=["alone"], inplace=True)

In [131]:
# FILL MISSING VALUES
df["age"] = df["age"].fillna(
    df["age"].median()
)

In [133]:
df = df.dropna(subset=["embarked"]) # Only two rows

In [135]:
df = pd.get_dummies(
    df,
    columns=["sex","embarked"],
    drop_first=True
)

In [137]:
df.to_csv(
    "data/processed/titanic_processed.csv",
    index=False
)